In [1]:
from agents import Agent, Runner
from dotenv import load_dotenv
from pydantic import BaseModel
import gradio as gr


c:\Users\dinga\Desktop\Agentic AI cause Prep\Agentic-Ai-project1-Interview-assistance-agent-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [9]:
from pydantic import BaseModel

class JobAnalysis(BaseModel):
    resume_skills: list[str]
    required_skills: list[str]
    matching_skills: list[str]
    missing_skills: list[str]

In [3]:
#Create first agent


agent = Agent(
    name ="Job-analysis agent",
    instructions = 
    """
    Analyzes the student’s resume and target job description,

    Extracts skills, education, projects, and work experience,

    Identifies job requirements and qualifications,

    Finds matches and potential skill gaps""",

    model ="gpt-4o-mini"
)

In [ ]:

resume = """ 
I am a dot net developer with 5+ years expeirence,
I am profficient in C# and react for front end, also SQU and Mongo db database

"""
job_description = """
 we are looking for a dot net developer with 4+ years of expirience also with anglular expert
"""
input_string =  f""" resume:
{resume} 
job description:
 {job_description}.
 """

In [5]:
result1 = await Runner.run(agent, input_string)
print(result1.final_output)

Let's analyze the provided resume and job description:

### **Resume Analysis:**

**Skills:**
- Proficient in C#
- Proficient in React for front-end development
- Experience with SQL databases
- Experience with MongoDB

**Experience:**
- 5+ years as a .NET developer

### **Job Description Requirements:**

**Qualifications:**
- .NET developer with 4+ years of experience
- Expertise in Angular
  
### **Matches:**
- The candidate meets the experience requirement (5+ years).
- The candidate has relevant skills in .NET (C#).

### **Skill Gaps:**
- The job requires expertise in Angular, which the candidate does not mention.
- While the candidate has skills in React, it does not align with the requirement for Angular.

### **Summary:**
- The candidate largely meets the experience and a primary skill requirement but lacks specific knowledge in Angular as required by the job description. It would be beneficial for the candidate to either gain experience in Angular or highlight any transferable 

In [6]:
agent =Agent[any](
    name="Interview-agent",
    instructions="""
    create interview questions,
    ask personalized questions to the job seaker
    
    """,
    model ="gpt-4o-mini"
)

In [1]:
result = await Runner.run(agent, "ASP.NET interview questions")
print(result.final_output)

NameError: name 'Runner' is not defined

In [1]:
import os
import uuid
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv


# ============================================================
# SETUP
# ============================================================

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file.")

client = OpenAI(api_key=api_key)


# ============================================================
# TEXT TO SPEECH
# AI reads interview questions aloud
# ============================================================

def text_to_speech(text):

    try:
        audio_file = os.path.abspath(
            f"question_{uuid.uuid4().hex}.mp3"
        )

        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="alloy",
            input=text
        ) as response:
            response.stream_to_file(audio_file)

        return audio_file

    except Exception as e:
        print("TTS Error:", e)
        return None


# ============================================================
# START INTERVIEW
# ============================================================

def start_interview(resume, job_description):

    if resume is None:
        return (
            "Please upload your resume.",
            None,
            "❌ Resume missing"
        )

    if job_description is None:
        return (
            "Please upload the job description.",
            None,
            "❌ Job description missing"
        )

    question = (
        "Welcome to your interview. "
        "Please tell me about yourself and explain how your "
        "experience relates to this position."
    )

    audio = text_to_speech(question)

    return (
        question,
        audio,
        "🟢 Interview started"
    )


# ============================================================
# TRANSCRIBE CANDIDATE AUDIO
# ============================================================

def transcribe_audio(audio_file):

    if audio_file is None:
        return None

    try:

        with open(audio_file, "rb") as audio:

            transcription = client.audio.transcriptions.create(
                model="gpt-4o-mini-transcribe",
                file=audio
            )

        return transcription.text

    except Exception as e:
        print("Transcription Error:", e)
        return None


# ============================================================
# SUBMIT ANSWER
# ============================================================

def submit_answer(audio_file):

    if audio_file is None:
        return (
            "",
            "❌ Record your answer first."
        )

    transcript = transcribe_audio(audio_file)

    if transcript is None:
        return (
            "",
            "❌ Could not transcribe recording."
        )

    return (
        transcript,
        "✅ Answer received"
    )


# ============================================================
# GET AI FEEDBACK
# ============================================================

def get_feedback(question, answer):

    if not answer or not answer.strip():

        return (
            "Please record and submit your answer first.",
            "❌ No answer available"
        )

    try:

        prompt = f"""
You are an AI Interview Coach.

Evaluate the candidate's interview answer.

Interview Question:
{question}

Candidate Answer:
{answer}

Give short and useful interview feedback.

Evaluate:

1. Relevance
2. Clarity
3. Technical/content quality
4. Communication
5. What was done well
6. What should be improved

Then give an improved example answer.

Keep the feedback concise and practical.
"""

        response = client.responses.create(
            model="gpt-4o-mini",
            input=prompt
        )

        feedback = response.output_text

        return (
            feedback,
            "⭐ Feedback generated"
        )

    except Exception as e:

        print("Feedback Error:", e)

        return (
            "Unable to generate feedback.",
            "❌ Feedback error"
        )


# ============================================================
# COMPACT CSS
# ============================================================

css = """

.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    padding: 6px !important;
}

h1 {
    font-size: 24px !important;
    margin: 2px !important;
}

h2 {
    font-size: 17px !important;
    margin: 2px !important;
}

footer {
    display: none !important;
}

"""


# ============================================================
# USER INTERFACE
# ============================================================

with gr.Blocks(
    title="AI Interview Coach",
    css=css
) as demo:

    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        """
        # 🎤 AI Interview Coach
        **Upload → Start → Listen → Record → Submit → Feedback**
        """
    )


    # ========================================================
    # DOCUMENT UPLOAD
    # ========================================================

    with gr.Row():

        resume_file = gr.File(
            label="📄 Resume",
            file_types=[
                ".pdf",
                ".docx",
                ".txt"
            ],
            type="filepath",
            height=65
        )

        jd_file = gr.File(
            label="📋 Job Description",
            file_types=[
                ".pdf",
                ".docx",
                ".txt"
            ],
            type="filepath",
            height=65
        )

        start_button = gr.Button(
            "▶ Start Interview",
            variant="primary"
        )


    # ========================================================
    # MAIN INTERVIEW AREA
    # ========================================================

    with gr.Row():

        # ----------------------------------------------------
        # LEFT SIDE - AI INTERVIEWER
        # ----------------------------------------------------

        with gr.Column(scale=1):

            gr.Markdown("## 🤖 Interviewer")

            question_output = gr.Textbox(
                label="Question",
                lines=2,
                max_lines=3,
                interactive=False
            )

            question_voice = gr.Audio(
                label="🔊 Question Audio",
                autoplay=True,
                interactive=False
            )


        # ----------------------------------------------------
        # RIGHT SIDE - CANDIDATE
        # ----------------------------------------------------

        with gr.Column(scale=1):

            gr.Markdown("## 🎙️ Your Answer")

            voice_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Record Answer"
            )

            submit_button = gr.Button(
                "✅ Submit Answer",
                variant="primary"
            )


    # ========================================================
    # TRANSCRIPT
    # ========================================================

    with gr.Row():

        transcript_output = gr.Textbox(
            label="📝 Your Answer",
            lines=2,
            max_lines=3,
            interactive=False
        )

        status_output = gr.Textbox(
            label="Status",
            lines=1,
            interactive=False
        )


    # ========================================================
    # FEEDBACK BUTTON
    # ========================================================

    feedback_button = gr.Button(
        "⭐ Get Interview Feedback",
        variant="secondary"
    )


    # ========================================================
    # FEEDBACK OUTPUT
    # ========================================================

    feedback_output = gr.Textbox(
        label="🧠 AI Coach Feedback",
        placeholder="Your interview feedback will appear here...",
        lines=5,
        max_lines=7,
        interactive=False
    )


    # ========================================================
    # EVENTS
    # ========================================================

    start_button.click(
        fn=start_interview,

        inputs=[
            resume_file,
            jd_file
        ],

        outputs=[
            question_output,
            question_voice,
            status_output
        ]
    )


    # ========================================================
    # SUBMIT ANSWER
    # ========================================================

    submit_button.click(
        fn=submit_answer,

        inputs=[
            voice_input
        ],

        outputs=[
            transcript_output,
            status_output
        ]
    )


    # ========================================================
    # GET FEEDBACK
    # ========================================================

    feedback_button.click(
        fn=get_feedback,

        inputs=[
            question_output,
            transcript_output
        ],

        outputs=[
            feedback_output,
            status_output
        ]
    )


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":

    demo.launch(
        inbrowser=True
    )

c:\Users\dinga\Desktop\Agentic AI cause Prep\Agentic-Ai-project1-Interview-assistance-agent-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\dinga\AppData\Local\Temp\ipykernel_23188\1154768386.py:231: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
